In [0]:
from pyspark.sql.functions import col, sum, when

orders_path = (
    "abfss://silver@ecommercenidhi.dfs.core.windows.net/orders"
)

orders = (
    spark.read
    .format("delta")
    .load(orders_path)
)

print("Total rows:", orders.count())
display(orders.limit(10))

In [0]:
null_ids = orders.filter(
    col("order_id").isNull() |
    col("customer_id").isNull() |
    col("product_id").isNull()
).count()

print("Rows with null IDs:", null_ids)

In [0]:
duplicate_orders = (
    orders
    .groupBy("order_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate order IDs:", duplicate_orders)

In [0]:
invalid_amounts = orders.filter(
    col("amount").isNull() |
    (col("amount") <= 0)
).count()

print("Invalid amounts:", invalid_amounts)

In [0]:
invalid_quantity = orders.filter(
    col("quantity").isNull() |
    (col("quantity") <= 0)
).count()

print("Invalid quantities:", invalid_quantity)

In [0]:
valid_statuses = ["completed", "pending", "cancelled"]

invalid_status = orders.filter(
    ~col("status").isin(valid_statuses)
).count()

print("Invalid statuses:", invalid_status)

In [0]:
quality_results = {
    "null_ids": null_ids,
    "duplicate_order_ids": duplicate_orders,
    "invalid_amounts": invalid_amounts,
    "invalid_quantities": invalid_quantity,
    "invalid_statuses": invalid_status
}

for check, result in quality_results.items():
    print(f"{check}: {result}")

In [0]:
import builtins

failed_checks = builtins.sum(
    1 for result in quality_results.values()
    if result > 0
)

if failed_checks > 0:
    raise Exception(
        f"Data Quality FAILED: {failed_checks} check(s) failed"
    )

print("Data Quality PASSED")

In [0]:
for check, result in quality_results.items():
    status = "FAILED" if result > 0 else "PASSED"
    print(f"{check}: {result} → {status}")

In [0]:
bad_amounts = orders.filter(
    col("amount").isNull() | (col("amount") <= 0)
)

bad_quantities = orders.filter(
    col("quantity").isNull() | (col("quantity") <= 0)
)

print("Bad amount records:")
display(bad_amounts)

print("Bad quantity records:")
display(bad_quantities)

In [0]:
from pyspark.sql.functions import when, lit

rejected_orders = (
    orders
    .filter(
        col("amount").isNull() |
        (col("amount") <= 0) |
        col("quantity").isNull() |
        (col("quantity") <= 0)
    )
    .withColumn(
        "rejection_reason",
        when(
            col("amount").isNull() | (col("amount") <= 0),
            lit("INVALID_AMOUNT")
        )
        .when(
            col("quantity").isNull() | (col("quantity") <= 0),
            lit("INVALID_QUANTITY")
        )
    )
)

display(rejected_orders)

In [0]:
print("Rejected records:", rejected_orders.count())

In [0]:
valid_orders = (
    orders
    .filter(
        col("amount").isNotNull() &
        (col("amount") > 0) &
        col("quantity").isNotNull() &
        (col("quantity") > 0)
    )
)

print("Valid records:", valid_orders.count())
print("Rejected records:", rejected_orders.count())

In [0]:
rejected_path = (
    "abfss://silver@ecommercenidhi.dfs.core.windows.net/"
    "quarantine/orders"
)

rejected_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .save(rejected_path)

In [0]:
display(
    spark.read
    .format("delta")
    .load(rejected_path)
)

In [0]:
rejected_path = (
    "abfss://silver@ecommercenidhi.dfs.core.windows.net/"
    "quarantine/orders"
)

rejected_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .save(rejected_path)

In [0]:
display(
    spark.read
    .format("delta")
    .load(rejected_path)
)

In [0]:
customers = (
    spark.read
    .format("delta")
    .load("abfss://silver@ecommercenidhi.dfs.core.windows.net/customers")
)

products = (
    spark.read
    .format("delta")
    .load("abfss://silver@ecommercenidhi.dfs.core.windows.net/products")
)

In [0]:
invalid_customers = (
    orders
    .join(
        customers.select("customer_id"),
        "customer_id",
        "left_anti"
    )
    .count()
)

invalid_products = (
    orders
    .join(
        products.select("product_id"),
        "product_id",
        "left_anti"
    )
    .count()
)

print("Orders with invalid customers:", invalid_customers)
print("Orders with invalid products:", invalid_products)